# 공정 조건별 품질 패턴 도출을 통한 제어 전략 제안

- 성결대학교 주관 데이터사이언스 경진대회 출품 프로젝트
- 실제 기업 공정 데이터를 기반으로 수행
- 원본 데이터는 기업 및 공정 관련 민감 정보로 인해 공개하지 않음
- 본 노트북은 발표 자료와 분석 결과를 바탕으로 재구성한 포트폴리오용 분석 코드임

## 분석 목표
- 품질 기준 변수 선정
- 품질 그룹별 공정 변수 차이 비교
- 공정 변수 간 상관관계 분석
- 제어 전략 도출

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")

## 1. 데이터 불러오기
Process Data와 QC Data를 불러온다.

In [7]:
DATA_DIR = Path("../data")

process_df = pd.read_excel(DATA_DIR / "Process_Data.xlsx")
qc_df = pd.read_excel(DATA_DIR / "QC_Data.xlsx")

process_df.columns = process_df.columns.str.strip()
qc_df.columns = qc_df.columns.str.strip()

print("Process Data shape:", process_df.shape)
print("QC Data shape:", qc_df.shape)

display(process_df.head())
display(qc_df.head())

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\Process_Data.xlsx'

## 2. 데이터 전처리 및 병합
날짜 기준으로 Process Data와 QC Data를 연결한다.

In [ ]:
# 실제 컬럼명에 맞게 수정 가능
process_date_col = "Date"
qc_date_col = "Date"
quality_col = "Weight"

process_df[process_date_col] = pd.to_datetime(process_df[process_date_col], errors="coerce").dt.date
qc_df[qc_date_col] = pd.to_datetime(qc_df[qc_date_col], errors="coerce").dt.date

merged_df = pd.merge(
    process_df,
    qc_df[[qc_date_col, quality_col]],
    left_on=process_date_col,
    right_on=qc_date_col,
    how="inner"
)

print("Merged Data shape:", merged_df.shape)
display(merged_df.head())

## 3. 품질 기준 설정
품질 지표로 `Weight`를 사용하고, 평균값을 기준으로 Good / Bad 그룹을 나눈다.

In [ ]:
quality_mean = merged_df[quality_col].mean()

merged_df["quality_group"] = np.where(
    merged_df[quality_col] >= quality_mean,
    "Good",
    "Bad"
)

print(f"{quality_col} 평균값: {quality_mean:.2f}")
print(merged_df["quality_group"].value_counts())
display(merged_df[[quality_col, "quality_group"]].head())

## 4. 품질 그룹별 주요 공정 변수 비교
발표 자료에서 핵심적으로 다룬 `ReactA_Temp`, `ReactB_Temp`를 중심으로 그룹별 평균을 비교한다.

In [ ]:
key_process_vars = ["ReactA_Temp", "ReactB_Temp"]

group_mean = (
    merged_df.groupby("quality_group")[key_process_vars]
    .mean()
    .T
    .reset_index()
)

group_mean.columns = ["variable", "Bad", "Good"]
display(group_mean)

In [ ]:
ax = group_mean.set_index("variable")[["Bad", "Good"]].plot(
    kind="bar",
    figsize=(8, 5)
)

plt.title("품질 그룹별 주요 공정 변수 평균 비교")
plt.xlabel("공정 변수")
plt.ylabel("평균값")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. 그룹별 차이 해석
Good / Bad 품질 그룹에서 어떤 온도 변수가 더 높게 나타나는지 확인한다.

In [8]:
for var in key_process_vars:
    good_mean = merged_df.loc[merged_df["quality_group"] == "Good", var].mean()
    bad_mean = merged_df.loc[merged_df["quality_group"] == "Bad", var].mean()

    print(f"[{var}]")
    print(f" - Good 평균: {good_mean:.2f}")
    print(f" - Bad 평균: {bad_mean:.2f}")

    if good_mean > bad_mean:
        print(f" -> {var}는 Good 그룹에서 상대적으로 높게 나타남\n")
    else:
        print(f" -> {var}는 Bad 그룹에서 상대적으로 높게 나타남\n")

NameError: name 'key_process_vars' is not defined

## 6. 공정 변수 간 상관관계 분석
공정 변수 간의 상관관계를 확인하여 중복 제어 가능성과 독립 제어 가능성을 탐색한다.

In [9]:
numeric_cols = merged_df.select_dtypes(include=[np.number]).columns.tolist()
corr_df = merged_df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("공정 변수 간 상관관계 히트맵")
plt.tight_layout()
plt.show()

NameError: name 'merged_df' is not defined

## 7. 결론
- `Weight`를 품질 기준으로 설정하여 Good / Bad 그룹을 구분하였다.
- 주요 온도 변수(`ReactA_Temp`, `ReactB_Temp`)는 품질 그룹별 평균 차이를 보였다.
- 상관관계 분석을 통해 일부 변수는 함께 움직이는 경향이 있음을 확인하였다.
- 이를 바탕으로 중복 제어를 줄이고 효율적인 공정 제어 전략을 제안할 수 있다.
- 향후 머신러닝 기반 품질 예측 모델로 확장 가능한 기반을 마련하였다.